---
<a id="monostack"></a>
## 08 — Monotonic Stack

> 📖 Review [monotonic_stack_basics.ipynb](../Basics/monotonic_stack_basics.ipynb) first!

| # | File | Difficulty | Key idea |
|---|------|------------|----------|
| [0109](#ms-0109) | [final_prices_discount.ipynb](../0109.final_prices_discount.ipynb) | 🟢 | Next smaller-or-equal, copy input |
| [0101](#ms-0101) | [daily_temperatures.ipynb](../0101.daily_temperatures.ipynb) | 🟡 | Store indices, `ans[idx] = i − idx` |
| [0103](#ms-0103) | [next_greater_element_single_list.ipynb](../0103.next_greater_element_single_list.ipynb) | 🟡 | Stack of indices, write on pop |
| [0107](#ms-0107) | [next_smaller_element.ipynb](../0107.next_smaller_element.ipynb) | 🟡 | Increasing stack — val < top to pop |
| [0108](#ms-0108) | [previous_greater_element.ipynb](../0108.previous_greater_element.ipynb) | 🟡 | Right-to-left scan |
| [0102](#ms-0102) | [next_greater_element.ipynb](../0102.next_greater_element.ipynb) | 🟡 | Two lists — lookup map value→index |
| [0104](#ms-0104) | [next_greater_element_circular.ipynb](../0104.next_greater_element_circular.ipynb) | 🟡 | `range(2*n), i%n`, push only if `i<n` |
| [0110](#ms-0110) | [online_stock_span.ipynb](../0110.online_stock_span.ipynb) | 🟡 | `(price, span)` tuples, absorb spans on pop |
| [0111](#ms-0111) | [car_fleet.ipynb](../0111.car_fleet.ipynb) | 🟡 | Sort by dist, push if new fleet |
| [0113](#ms-0113) | [next_greater_node_linked_list.ipynb](../0113.next_greater_node_linked_list.ipynb) | 🟡 | Flatten list → NGE stack |
| [0143](#ms-0143) | [largest_rectangle_histogram.ipynb](../0143.largest_rectangle_histogram.ipynb) | 🔴 | `(start, height)` tuples — pop when shorter bar arrives |

# Monotonic Stacks

## What is a Monotonic Stack?

A stack where elements are always maintained in a strictly increasing or decreasing order from bottom to top. When a new element violates that order, you **pop until the invariant is restored**, then push.

---

## The Core Technique

```
for each element x in array:
    while stack is not empty AND condition(stack.top, x):
        pop from stack        ← process the popped element here
    push x
```

The condition in the `while` determines which flavor you're using.

---

## Monotonic Increasing Stack

**Stack order (bottom → top):** small → large

**Pop condition:** `while stack.top > current` (pop anything **larger** than current)

```python
stack = []
for x in arr:
    while stack and stack[-1] > x:
        stack.pop()
    stack.append(x)
```

**What it finds:** For each element, the **next smaller element** to its left (what survived the pops).

**Use when the problem asks about:**
- Next Smaller Element
- Previous Smaller Element
- Largest Rectangle in Histogram (spans between smaller boundaries)
- Trap Rain Water

---

## Monotonic Decreasing Stack

**Stack order (bottom → top):** large → small

**Pop condition:** `while stack.top < current` (pop anything **smaller** than current)

```python
stack = []
for x in arr:
    while stack and stack[-1] < x:
        stack.pop()
    stack.append(x)
```

**What it finds:** For each element, the **next greater element** to its left.

**Use when the problem asks about:**
- Next Greater Element
- Daily Temperatures (next warmer day)
- Stock Span Problem
- Maximum width ramps

---

## Quick Decision Table

| You need… | Stack type | Pop when… |
|---|---|---|
| Next **smaller** to right | Increasing | `top > current` |
| Next **greater** to right | Decreasing | `top < current` |
| Previous **smaller** to left | Increasing | process on push |
| Previous **greater** to left | Decreasing | process on push |

---

## The Key Insight

> **When you pop an element, you've just found its answer.**

The element being popped (`top`) now knows: *"the current element x is the first one to my right that broke my invariant"* — that's exactly the next smaller/greater it was looking for.

```python
# Next Greater Element pattern
result = [-1] * len(arr)
stack = []  # stores indices

for i, x in enumerate(arr):
    while stack and arr[stack[-1]] < x:
        idx = stack.pop()
        result[idx] = x      # ← x is the next greater for arr[idx]
    stack.append(i)
```

---

## Memory Aid

```
Increasing stack  →  guards against small intruders  →  finds SMALLER neighbors
Decreasing stack  →  guards against large intruders  →  finds GREATER neighbors
```

<a id="ms-0109"></a>
# 0109 Final Prices With a Special Discount in a Shop
[↑ Back to TOC](#monostack)


In [1]:
"""
id: lc_1475
title: Final Prices With a Special Discount in a Shop
source: leetcode
difficulty: easy
primary: monotonic stack
tags: [array, stack, monotonic-stack]
leetcode_url: https://leetcode.com/problems/final-prices-with-a-special-discount-in-a-shop/
status: draft
last_updated: 2026-04-23
notes:
- key idea: use a monotonic stack to find the first smaller or equal element to the right
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 1475_lc_1475_final_prices_with_a_special_discount_in_a_shop_empty.py
# LeetCode 1475: Final Prices With a Special Discount in a Shop
# Difficulty: Easy
#
# PROBLEM STATEMENT:
# Given the array prices, where prices[i] is the price of the ith item in a shop.
# There is a special discount for items in the shop. If you buy the ith item, 
# then you will receive a discount equivalent to prices[j] where j is the 
# minimum index such that j > i and prices[j] <= prices[i]. Otherwise, you 
# will not receive any discount at all.
# Return an array where the ith element is the final price you will pay for 
# the ith item of the shop, considering the special discount.
#
# RULES:
# - Discount is applied if prices[j] <= prices[i] for the first j > i.
# - Final price = prices[i] - prices[j].
#
# EXAMPLES:
# Input: prices = [8,4,6,2,3]
# Output: [4,2,4,2,3]
#
# Input: prices = [1,2,3,4,5]
# Output: [1,2,3,4,5]
# ============================================================================

from typing import List

def finalPrices(prices: List[int]) -> List[int]:
    stack =  []               # mono increasing stack
    n = len(prices)
    out = prices[:]

    for i, price in enumerate(prices):
        while stack and price <= prices[stack[-1]]:
            j = stack.pop()
            out[j] = prices[j] - price
        stack.append(i)
    return out
print(finalPrices([8,4,6,2,3]))
print(finalPrices([1,2,3,4,5]))

def test():
    assert finalPrices([8,4,6,2,3]) == [4,2,4,2,3]  # given example 1
    assert finalPrices([1,2,3,4,5]) == [1,2,3,4,5]  # given example 2 - increasing
    assert finalPrices([10,1,1,6]) == [9,0,1,6]      # given example 3 - duplicate mins
    assert finalPrices([5,4,3,2,1]) == [1,1,1,1,1]  # strictly decreasing
    assert finalPrices([10,2,5,2,8]) == [8,0,3,2,8] # non-monotonic mix
    assert finalPrices([1]) == [1]                  # single element
    assert finalPrices([2,2,2]) == [0,0,2]          # identical elements
    assert finalPrices([10,9]) == [1,9]             # basic two element drop
    assert finalPrices([9,10]) == [9,10]            # basic two element rise
    assert finalPrices([]) == []                    # empty input
    print('All Pass!')

test()

[4, 2, 4, 2, 3]
[1, 2, 3, 4, 5]
All Pass!


<a id="ms-0101"></a>
# 0101 Daily Temperatures
[↑ Back to TOC](#monostack)


In [2]:
"""
id: lc_0739
title: Daily Temperatures
source: leetcode
difficulty: medium
primary: monotonic-stack
tags: [stack, array]
leetcode_url: https://leetcode.com/problems/daily-temperatures/
status: draft
last_updated: 2026-04-23
notes:
- key idea: use a monotonic decreasing stack to track indices of temperatures needing a warmer day
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 739_lc_0739_daily_temperatures_empty.py
# LeetCode 739: Daily Temperatures
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given an array of integers temperatures, return an array answer where 
# answer[i] is the number of days you have to wait after day i to get a 
# warmer temperature. If there is no future warmer day, answer[i] = 0.
#
# RULES:
# - An O(n) solution is expected.
# - Input array length is between 1 and 10^5.
# - Temperature values are between 30 and 100.
#
# EXAMPLES:
# Input: temperatures = [73,74,75,71,69,72,76,73]
# Output: [1,1,4,2,1,1,0,0]
# ============================================================================

from typing import List

def dailyTemperatures(temperatures: List[int]) -> List[int]:
    stack = []                                        #Mono Decreasing stack
    out = [0] * len(temperatures)
    for i, temp in enumerate(temperatures):
        while stack and temp > temperatures[stack[-1]]:
            ind = stack.pop()
            out[ind] = i - ind
        stack.append(i)
    return out


print(dailyTemperatures([73, 74, 75, 71, 69, 72, 76, 73]))
print(dailyTemperatures([30, 40, 50, 60]))

def test():
    assert dailyTemperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]  # given example 1
    assert dailyTemperatures([30, 40, 50, 60]) == [1, 1, 1, 0]  # given example 2
    assert dailyTemperatures([30, 60, 90]) == [1, 1, 0]  # given example 3
    assert dailyTemperatures([89, 62, 70, 58, 47, 47, 46, 76, 100, 70]) == [8, 1, 5, 4, 3, 2, 1, 1, 0, 0]  # mixed variation
    assert dailyTemperatures([30, 30, 30]) == [0, 0, 0]  # no increase
    assert dailyTemperatures([50, 40, 30]) == [0, 0, 0]  # strictly decreasing
    assert dailyTemperatures([30, 40, 30]) == [1, 0, 0]  # local peak
    assert dailyTemperatures([100]) == [0]  # single element
    assert dailyTemperatures([30, 100]) == [1, 0]  # two elements increase
    assert dailyTemperatures([100, 30]) == [0, 0]  # two elements decrease
    print('All Pass!')

test()

[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
All Pass!


<a id="ms-0103"></a>
# 0103 Next Greater Element I
[↑ Back to TOC](#monostack)


In [3]:
"""
id: lc_0496
title: Next Greater Element I
source: leetcode
difficulty: easy
primary: stack
tags: [monotonic-stack, array, hash-table]
leetcode_url: https://leetcode.com/problems/next-greater-element-i/
status: draft
last_updated: 2026-04-23
notes:
- key idea: Monotonic decreasing stack to find the first larger value.
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 496_lc_next_greater_element_i_empty.py
# LeetCode 496: Next Greater Element I
# Difficulty: Easy
#
# PROBLEM STATEMENT:
# For each element in an array nums, find the first element to its right 
# that is strictly greater than it. If it does not exist, return -1.
#
# RULES:
# - Return an array of the same length.
# - Search is only to the right (not circular).
#
# EXAMPLES:
# [2, 1, 2, 4, 3] -> [4, 2, 4, -1, -1]
# [1, 2, 3]       -> [2, 3, -1]
# ============================================================================

from typing import List

def nextGreaterElement(nums: List[int]) -> List[int]:
    n  = len(nums)
    out = [-1] * n
    stack = []                      #mono decreasing stack
    for i, num in enumerate (nums):
        while stack and num > nums[stack[-1]]:
            ind = stack.pop()
            out[ind] = nums[i]
        stack.append(i)
    return (out)


print(nextGreaterElement([2, 1, 2, 4, 3]))
print(nextGreaterElement([1, 2, 3]))

def test():
    assert nextGreaterElement([2, 1, 2, 4, 3]) == [4, 2, 4, -1, -1]  # given example 1
    assert nextGreaterElement([1, 2, 3]) == [2, 3, -1]              # given example 2
    assert nextGreaterElement([3, 2, 1]) == [-1, -1, -1]           # given example 3
    assert nextGreaterElement([1, 5, 3, 4, 2]) == [5, -1, 4, -1, -1] # multi-peak
    assert nextGreaterElement([1]) == [-1]                         # single element
    assert nextGreaterElement([]) == []                             # empty case
    assert nextGreaterElement([1, 1, 1]) == [-1, -1, -1]           # duplicate values (non-greater)
    assert nextGreaterElement([10, 5, 20]) == [20, 20, -1]         # immediate vs distant greater
    assert nextGreaterElement([4, 3, 2, 5]) == [5, 5, 5, -1]       # all point to last
    print('All Pass!')

test()

[4, 2, 4, -1, -1]
[2, 3, -1]
All Pass!


<a id="ms-0107"></a>
# 0107 Next Smaller Element
[↑ Back to TOC](#monostack)


In [4]:
"""
id: lc_nse
title: Next Smaller Element
source: leetcode
difficulty: medium
primary: stack
tags: [monotonic-stack, array]
leetcode_url: https://leetcode.com/problems/next-smaller-element/
status: draft
last_updated: 2026-04-23
notes:
- key idea: Use a monotonic increasing stack, processing from right to left or left to right.
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 000_lc_nse_next_smaller_element_empty.py
# LeetCode: Next Smaller Element
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given an array nums, for each element return the first element to its right 
# that is smaller. If none exists, return -1.
#
# RULES:
# - Return an array of the same length.
# - If no smaller element exists to the right, use -1.
#
# EXAMPLES:
# [4, 2, 1, 5, 3] -> [2, 1, -1, 3, -1]
# [3, 2, 1]       -> [2, 1, -1]
# [1, 2, 3]       -> [-1, -1, -1]
# ============================================================================

from typing import List

def nextSmallerElement(nums: List[int]) -> List[int]:
    stack = [] # mono incresing stack
    n = len(nums)
    out = [-1] * n

    for i, num in enumerate(nums):
        while stack and num < nums[stack[-1]]:
            ind = stack.pop()
            out[ind] = num
        stack.append(i)
    return out
    

print(nextSmallerElement([4, 2, 1, 5, 3]))
print(nextSmallerElement([3, 2, 1]))

def test():
    assert nextSmallerElement([4, 2, 1, 5, 3]) == [2, 1, -1, 3, -1]  # given example 1
    assert nextSmallerElement([3, 2, 1]) == [2, 1, -1]              # strictly decreasing
    assert nextSmallerElement([1, 2, 3]) == [-1, -1, -1]           # strictly increasing
    assert nextSmallerElement([5, 5, 5]) == [-1, -1, -1]           # all same
    assert nextSmallerElement([1]) == [-1]                         # single element
    assert nextSmallerElement([]) == []                             # empty input
    assert nextSmallerElement([2, 1, 2, 1, 1]) == [1, -1, 1, -1, -1] # duplicates
    assert nextSmallerElement([10, 1, 1, 1]) == [1, -1, -1, -1]     # large jump
    assert nextSmallerElement([1, 10, 2]) == [-1, 2, -1]            # peak in middle
    print('All Pass!')

test()

[2, 1, -1, 3, -1]
[2, 1, -1]
All Pass!


<a id="ms-0108"></a>
# 0108 Previous Greater Element
[↑ Back to TOC](#monostack)


In [5]:
"""
id: lc_prev_greater
title: Previous Greater Element
source: leetcode
difficulty: medium
primary: monotonic-stack
tags: [stack, array]
leetcode_url: https://leetcode.com/problems/previous-greater-element/
status: draft
last_updated: 2026-04-23
notes:
- key idea: maintain a monotonic decreasing stack from left to right
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 000_lc_prev_greater_previous_greater_element_empty.py
# LeetCode: Previous Greater Element
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given an array nums, for each element return the first element to its left 
# that is greater. If none exists, return -1.
#
# RULES:
# - Maintain a decreasing stack.
# - Pop elements smaller than or equal to current.
# - The top of the stack is the previous greater element.
#
# EXAMPLES:
# Input: [4, 5, 2, 10, 8]
# Output: [-1, -1, 5, -1, 10]
# ============================================================================

from typing import List

def prevGreaterElement(nums: List[int]) -> List[int]:
    n = len(nums)
    stack = []              # mono decrasing stack
    out = [-1] * n
    for i in range(n-1, -1 , -1):
        num = nums[i]
        while stack and num > nums[stack[-1]]:
            ind = stack.pop()
            out[ind] = num
        stack.append(i)
    return out
        
    

print(prevGreaterElement([4, 5, 2, 10, 8]))
print(prevGreaterElement([1, 2, 3]))

def test():
    assert prevGreaterElement([4, 5, 2, 10, 8]) == [-1, -1, 5, -1, 10]  # example 1
    assert prevGreaterElement([1, 2, 3]) == [-1, -1, -1]               # strictly increasing
    assert prevGreaterElement([3, 2, 1]) == [-1, 3, 2]               # strictly decreasing
    assert prevGreaterElement([10, 10, 10]) == [-1, -1, -1]           # duplicate values
    assert prevGreaterElement([5]) == [-1]                            # single element
    assert prevGreaterElement([]) == []                               # empty list
    assert prevGreaterElement([2, 1, 5]) == [-1, 2, -1]               # local peak
    assert prevGreaterElement([1, 5, 2, 4]) == [-1, -1, 5, 5]         # mixed jumps
    assert prevGreaterElement([10, 2, 3, 4]) == [-1, 10, 10, 10]      # multiple smaller after big
    assert prevGreaterElement([1, 2, 1, 2]) == [-1, -1, 2, -1]        # oscillating
    print('All Pass!')

test()

[-1, -1, 5, -1, 10]
[-1, -1, -1]
All Pass!


<a id="ms-0102"></a>
# 0102	next_greater_element.ipynb
[↑ Back to TOC](#monostack)


In [6]:
"""
id: lc_0496
title: Next Greater Element I
source: leetcode
difficulty: easy
primary: stack
tags: [monotonic-stack, hash-table, array]
leetcode_url: https://leetcode.com/problems/next-greater-element-i/
status: draft
last_updated: 2026-04-23
notes:
- key idea: Monotonic decreasing stack to find next greater elements in nums2.
- time: O(n + m)
- space: O(m)
"""

# ============================================================================
# File: 496_lc_0496_next_greater_element_i_empty.py
# LeetCode 496: Next Greater Element I
# Difficulty: Easy
#
# PROBLEM STATEMENT:
# For each element in nums1, find the first element to its right in nums2 that 
# is larger than it. If it does not exist, return -1. 
# nums1 is a subset of nums2.
#
# RULES:
# - nums1 elements are unique.
# - nums2 elements are unique.
# - All elements in nums1 are present in nums2.
#
# EXAMPLES:
# nums1 = [4,1,2], nums2 = [1,3,4,2] -> [-1,3,-1]
# nums1 = [2,4], nums2 = [1,2,3,4] -> [3,-1]
# ============================================================================

from typing import List

def nextGreaterElement(nums1: List[int], nums2: List[int]) -> List[int]:
    lookup = {num: i for i, num in enumerate(nums1)}

    out = [-1]  * len(nums1)
    stack = []           #mono decreasing stack

    for i, num in enumerate(nums2):
        while stack and num > nums2[stack[-1]]:
            ind = stack.pop()
            popped_num = nums2[ind]
            if popped_num in lookup:
                out[lookup[popped_num]] = num
        stack.append(i)
    
    return out

print(nextGreaterElement([4,1,2], [1,3,4,2]))
print(nextGreaterElement([2,4], [1,2,3,4]))

def test():
    assert nextGreaterElement([4,1,2], [1,3,4,2]) == [-1,3,-1]  # example 1
    assert nextGreaterElement([2,4], [1,2,3,4]) == [3,-1]      # example 2
    assert nextGreaterElement([1,3,5,2,4], [6,5,4,3,2,1,7]) == [7,7,7,7,7]  # all point to end
    assert nextGreaterElement([1], [1]) == [-1]                # single element
    assert nextGreaterElement([1,2], [1,2]) == [2,-1]          # simple sequence
    assert nextGreaterElement([5,4,3], [5,4,3,6]) == [6,6,6]    # shared successor
    assert nextGreaterElement([4,2], [1,2,4,3]) == [-1,4]      # subset out of order
    assert nextGreaterElement([1,2,3], [3,2,1]) == [-1,-1,-1]  # descending order
    assert nextGreaterElement([10], [1,10,5]) == [-1]          # middle element no greater
    assert nextGreaterElement([1,5], [1,2,3,4,5]) == [2,-1]    # start and end
    print('All Pass!')

test()

[-1, 3, -1]
[3, -1]
All Pass!


<a id="ms-0104"></a>
# 0104	  Next Greater Element II Circular     for j in range(2 * n):         i = j % n
[↑ Back to TOC](#monostack)


In [7]:
"""
id: lc_0503
title: Next Greater Element II
source: leetcode
difficulty: medium
primary: monotonic stack
tags: [stack, array, circular-array]
leetcode_url: https://leetcode.com/problems/next-greater-element-ii/
status: draft
last_updated: 2026-04-23
notes:
- key idea: Monotonic decreasing stack while iterating 2n times to simulate circularity.
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 503_lc_0503_next_greater_element_ii_empty.py
# LeetCode 503: Next Greater Element II
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given a circular integer array nums, return the next greater number for every 
# element in nums. The next greater number of a number x is the first greater 
# number to its traversing-order next in the array, which means you could search 
# circularly to find its next greater number. If it doesn't exist, return -1.
#
# RULES:
# - Array wraps around: index n-1 looks at index 0.
# - Return -1 if no greater element exists.
#
# EXAMPLES:
# Input: [1,2,1] -> Output: [2,-1,2]
# Input: [3,1,2,4] -> Output: [4,2,4,-1]
# ============================================================================

from typing import List

def nextGreaterElements(nums: List[int]) -> List[int]:
    n = len(nums)
    out = [-1] * n
    stack = []  # decreasing stack of indices

    for j in range(2 * n):
        i = j % n
        while stack and nums[i] > nums[stack[-1]]:
            out[stack.pop()] = nums[i]
        if j < n:                 # push only in first pass
            stack.append(i)

    return out
print(nextGreaterElements([1, 2, 1]))
print(nextGreaterElements([3, 1, 2, 4]))

def test():
    assert nextGreaterElements([1, 2, 1]) == [2, -1, 2]  # example 1: basic circularity
    assert nextGreaterElements([3, 1, 2, 4]) == [4, 2, 4, -1]  # example 2: max element stays -1
    assert nextGreaterElements([1, 2, 3, 4, 3]) == [2, 3, 4, -1, 4]  # example 3: wrap around finds 4
    assert nextGreaterElements([5, 4, 3, 2, 1]) == [-1, 5, 5, 5, 5]  # strictly decreasing
    assert nextGreaterElements([1, 1, 1, 1]) == [-1, -1, -1, -1]  # all identical
    assert nextGreaterElements([1]) == [-1]  # single element
    assert nextGreaterElements([1, 5, 2, 4, 3]) == [5, -1, 4, 5, 5]  # mixed circularity
    assert nextGreaterElements([]) == []  # empty input edge case
    print('All Pass!')

test()

[2, -1, 2]
[4, 2, 4, -1]
All Pass!


<a id="ms-0110"></a>
# 0110	Online Stock Span
[↑ Back to TOC](#monostack)


In [13]:
"""
id: lc_0901
title: Online Stock Span
source: leetcode
difficulty: medium
primary: stack
tags: [monotonic-stack, design, data-stream]
leetcode_url: https://leetcode.com/problems/online-stock-span/
status: draft
last_updated: 2026-04-24
notes:
- key idea: monotonic decreasing stack of (price, span)
- time: O(1) average (O(N) total for N calls)
- space: O(N)
"""

# ============================================================================
# File: 901_lc_0901_online_stock_span_empty.py
# LeetCode 901: Online Stock Span
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Design a class StockSpanner which collects daily price quotes and returns the 
# span of that stock's price for the current day. The span is the maximum number 
# of consecutive days (starting from today and going backward) for which the 
# stock price was less than or equal to today's price.
#
# RULES:
# - next(price) returns the integer span.
# - The span includes the current day itself.
#
# EXAMPLES:
# Input: ["StockSpanner", "next", "next", "next", "next", "next", "next", "next"]
# [[], [100], [80], [60], [70], [60], [75], [85]]
# Output: [null, 1, 1, 1, 2, 1, 4, 6]
# ============================================================================

from typing import List

class StockSpanner:
    def __init__(self):
        self.data = []          #price and span   mono decreasing stack


    def next(self, price: int) -> int:
        span = 1
        while self.data and price >= self.data[-1][0]:
            _,  pevspan = self.data.pop()
            span += pevspan
        self.data.append((price, span))
        return span

spanner = StockSpanner()
print(spanner.next(100))
print(spanner.next(80))

def test():
    s = StockSpanner()
    assert s.next(100) == 1   # base case
    assert s.next(80) == 1    # price decreases
    assert s.next(60) == 1    # price decreases
    assert s.next(70) == 2    # absorbs 60
    assert s.next(60) == 1    # break streak
    assert s.next(75) == 4    # absorbs 60, 70, 60
    assert s.next(85) == 6    # absorbs everything since 100
    assert s.next(100) == 8   # equals first price, full span
    assert s.next(110) == 9   # exceeds all previous
    assert s.next(50) == 1    # small value reset
    print('All Pass!')

test()

1
1
All Pass!


<a id="ms-0111"></a>
# 0111	CarFleet
[↑ Back to TOC](#monostack)

In [14]:
"""
id: lc_0853
title: Car Fleet
source: leetcode
difficulty: medium
primary: stack
tags: [array, stack, sorting]
leetcode_url: https://leetcode.com/problems/car-fleet/
status: draft
last_updated: 2026-04-24
notes:
- key idea: sort cars by position descending and track arrival times; a car merges if its arrival time <= the car ahead
- time: O(n log n) due to sorting
- space: O(n) for sorting/stack
"""

# ============================================================================
# File: 853_lc_0853_car_fleet_empty.py
# LeetCode 853: Car Fleet
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# n cars are going to the same destination called target, which is target miles away.
# Each car i has a constant speed speed[i] (in miles per hour), and initial position
# position[i] miles from the target towards the destination.
#
# A car can never pass another car ahead of it, but it can catch up to it and 
# drive bumper to bumper at the same speed. The faster car will slow down to 
# match the slower car's speed. The distance between these two cars is ignored.
#
# A car fleet is some non-empty set of cars driving at the same position and 
# same speed. Note that a single car is also a car fleet.
#
# RULES:
# - If a car catches up to another car at the target mile, it is still 
#   considered as one car fleet.
# - Return the number of car fleets that will arrive at the destination.
#
# EXAMPLES:
# Input: target = 12, position = [10,8,0,5,3], speed = [2,4,1,1,3]
# Output: 3
# ============================================================================

from typing import List

def carFleet(target: int, position: List[int], speed: List[int]) -> int:
    n = len(position)
    if n < 2: return n
    dist_to_target = [target -x for x in position]
    cars = sorted(list(zip(dist_to_target, speed)) )
    arrival = [dist/ speed for dist, speed in cars]
    stack = []                 # mono increasing stack
    for t in arrival:          # closest → farthest
        while not stack or t > stack[-1]:   # only add if only the arrival time > than the arrived before. If less or Equal it is part of the fleet
            stack.append(t)    # merges into the fleet ahead
            
    return len(stack)

print(carFleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))
print(carFleet(10, [0, 4, 2], [2, 1, 3]))

def test():
    assert carFleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3  # given example 1
    assert carFleet(10, [3], [3]) == 1                          # single car
    assert carFleet(100, [0, 2, 4], [4, 2, 1]) == 1             # all merge into one
    
    assert carFleet(10, [6, 8], [3, 2]) == 2                    # starts behind, but doesn't catch up
    assert carFleet(10, [8, 6], [2, 3]) == 2                  # catches up at target
    assert carFleet(10, [2, 4], [3, 2]) == 1                    # catches up before target
    assert carFleet(12, [10, 8], [2, 4]) == 1                   # same arrival time
    assert carFleet(10, [0, 0, 0], [1, 1, 1]) == 1              # same position (if possible by constraints)
    assert carFleet(10, [5, 2, 3], [1, 1, 1]) == 3              # identical speeds, distinct positions
    assert carFleet(10, [0, 4, 2], [2, 1, 3]) == 1              # no cars catch up
    print('All Pass!')

test()

3
1
All Pass!


<a id="ms-0113"></a>
# 0113 Next Greater Node In Linked List (Placeholder)
[↑ Back to TOC](#monostack)


In [2]:
"""
id: lc_1019
title: Next Greater Node In Linked List
source: leetcode
difficulty: medium
primary: stack
tags: [linked-list, stack, monotonic-stack]
leetcode_url: https://leetcode.com/problems/next-greater-node-in-linked-list/
status: draft
last_updated: 2026-04-24
notes:
- key idea: convert to list then use monotonic stack
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 1019_lc_1019_next_greater_node_in_linked_list_empty.py
# LeetCode 1019: Next Greater Node In Linked List
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# You are given the head of a linked list with n nodes.
# For each node in the list, find the value of the next greater node. That is, 
# for the current node, find the value of the first node that is next to it 
# and has a strictly larger value than it.
# Return an integer array answer where answer[i] is the value of the next 
# greater node of the ith node (1-indexed). If the ith node does not have a 
# next greater node, set answer[i] = 0.
#
# RULES:
# - The number of nodes in the list is in the range [1, 10^4].
# - 1 <= Node.val <= 10^9
#
# EXAMPLES:
# Input: head = [2,1,5]
# Output: [5,5,0]
#
# Input: head = [2,7,4,3,5]
# Output: [7,0,5,5,0]
# ============================================================================

from typing import List, Optional

# Definition for singly-linked list.
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def nextGreaterNodes(head: Optional[ListNode]) -> List[int]:
    node = head 
    nums = []
    while node:
        nums.append(node.val)
        node = node.next
        
    n  = len(nums)
    out = [0] * n
    stack = []                      #mono decreasing stack
    for i, num in enumerate (nums):
        while stack and num > nums[stack[-1]]:
            ind = stack.pop()
            out[ind] = nums[i]
        stack.append(i)
    return (out)



# Helper to build linked list for demo
def build_ll(arr):
    if not arr: return None
    head = ListNode(arr[0])
    curr = head
    for v in arr[1:]:
        curr.next = ListNode(v)
        curr = curr.next
    return head

print(nextGreaterNodes(build_ll([2,1,5])))
print(nextGreaterNodes(build_ll([2,7,4,3,5])))

def test():
    assert nextGreaterNodes(build_ll([2,1,5])) == [5,5,0]  # example 1
    assert nextGreaterNodes(build_ll([2,7,4,3,5])) == [7,0,5,5,0]  # example 2
    assert nextGreaterNodes(build_ll([1,7,5,1,9,2,5,1])) == [7,9,9,9,0,5,0,0]  # longer list
    assert nextGreaterNodes(build_ll([5])) == [0]  # single element
    assert nextGreaterNodes(build_ll([1,2,3,4])) == [2,3,4,0]  # strictly increasing
    assert nextGreaterNodes(build_ll([4,3,2,1])) == [0,0,0,0]  # strictly decreasing
    assert nextGreaterNodes(build_ll([2,2,2])) == [0,0,0]  # all same
    assert nextGreaterNodes(build_ll([2,3,2])) == [3,0,0]  # peak in middle
    assert nextGreaterNodes(build_ll([3,1,2])) == [0,2,0]  # valley
    assert nextGreaterNodes(build_ll([1,5,2,5])) == [5,0,5,0]  # duplicate next greater
    print('All Pass!')

test()

[5, 5, 0]
[7, 0, 5, 5, 0]
All Pass!


<a id="ms-0143"></a>
# 0143 Largest Rectangle In Histogram (Placeholder)
[↑ Back to TOC](#monostack)


In [3]:
"""
id: lc_0084
title: Largest Rectangle in Histogram
source: leetcode
difficulty: hard
primary: monotonic stack
tags: [stack, array, monotonic-stack]
leetcode_url: https://leetcode.com/problems/largest-rectangle-in-histogram/
status: draft
last_updated: 2026-04-24
notes:
- key idea: Monotonic increasing stack of (index, height)
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 084_lc_0084_largest_rectangle_in_histogram_empty.py
# LeetCode 84: Largest Rectangle in Histogram
# Difficulty: Hard
#
# PROBLEM STATEMENT:
# Given an array of integers heights representing the histogram's bar height 
# where the width of each bar is 1, return the area of the largest rectangle 
# in the histogram.
#
# RULES:
# - Heights are non-negative.
# - You must find the maximum area possible.
#
# EXAMPLES:
# Input: heights = [2,1,5,6,2,3]
# Output: 10 (The rectangle is formed by heights 5 and 6 with area 2 * 5 = 10)
# ============================================================================

from typing import List

def largestRectangleArea(heights: List[int]) -> int:
    best = 0 
    stack = []                  #mono increasing stack ( h , i)
    n = len(heights)
    for i , h in enumerate(heights):
        inh_ind = i
        while stack and h < stack[-1][0]:
            popped_height, popped_ind = stack.pop()
            inh_ind = popped_ind
            best = max(best, popped_height * (i -popped_ind ))
        stack.append((h, inh_ind ))
    while stack:
        popped_height, popped_ind = stack.pop()
        best = max(best, popped_height * (n - popped_ind))

    return best
        
        
    

print(largestRectangleArea([2, 1, 5, 6, 2, 3]))
print(largestRectangleArea([2, 4]))

def test():
    assert largestRectangleArea([2, 1, 5, 6, 2, 3]) == 10  # standard case
    assert largestRectangleArea([2, 4]) == 4              # two bars increasing
    assert largestRectangleArea([1, 1]) == 2              # uniform height
    assert largestRectangleArea([0, 9]) == 9              # zero height presence
    assert largestRectangleArea([4, 2, 0, 3, 2, 5]) == 6  # disconnected components
    assert largestRectangleArea([5, 4, 3, 2, 1]) == 9     # decreasing sequence
    assert largestRectangleArea([1, 2, 3, 4, 5]) == 9     # increasing sequence
    assert largestRectangleArea([2, 1, 2]) == 3           # valley shape
    assert largestRectangleArea([1]) == 1                 # single element
    assert largestRectangleArea([]) == 0                  # empty list
    print('All Pass!')

test()

10
4
All Pass!
